# Agentic Retail Capstone: SFT, RFT, MCP, and Policy Evaluation

**Role:** mixed capstone | **Level:** advanced | **Default mode:** local validation and analysis only

Integrate a fictional retail agent's policy, tools, immutable data snapshots, next-tool evaluation, supervised fine-tuning (SFT), policy-reward RFT, MCP/remote endpoints, and final agent wiring. The notebook intentionally creates no cloud resources, agents, evaluations, jobs, deployments, or database mutations.

## Architecture and learning goals

The complete system is a versioned bundle, not just a model:

1. **Policy and retail state** define allowed behavior and ground truth.
2. **Tool schemas** expose read and mutation operations.
3. **MCP** supports interactive agent discovery/execution; Agentic RFT uses explicit remote HTTP tool bindings during rollouts.
4. **SFT** imitates high-quality traces and stabilizes short tool sequences.
5. **RFT** optimizes whole-rollout policy correctness with a grader.
6. **Held-out evaluation** compares base, SFT, and RFT candidates under the same contract.

Prerequisites: core SFT, deterministic evaluation, core RFT, and the preceding tool-calling lesson.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
COOKBOOKS_ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(COOKBOOKS_ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)

from cookbook_utils import (
    configure_logging,
    env_flag,
    evaluate_next_tool_calls,
    load_json,
    load_jsonl,
    normalize_tool_call,
    require_env,
    resolve_tool_endpoints,
    sha256_file,
    validate_conversation_rows,
    validate_exact_generated_output,
)
from shared import (
    create_or_reuse_fine_tuning_job,
    create_project_context,
    load_foundry_config,
    monitor_fine_tuning_job,
    upload_or_reuse_file,
)
from shared.deployment import ensure_deployment, invoke_chat

configure_logging()
DATA = ROOT / "data"
CONFIGS = ROOT / "configs"
ASSETS = ROOT / "assets"
RESULTS = ROOT / "assets" / "metrics"

RUN_PAID_SFT = env_flag("FOUNDRY_RUN_PAID_SFT")
RUN_PAID_RFT = env_flag("FOUNDRY_RUN_PAID_RFT")
RUN_LIVE_EVALUATION = env_flag("FOUNDRY_RUN_LIVE_EVALUATION")
USE_LIVE_GENERATED_OUTPUT = env_flag("FOUNDRY_USE_LIVE_GENERATED_OUTPUT")
FILE_TIMEOUT_SECONDS = int(os.getenv("FOUNDRY_FILE_TIMEOUT_SECONDS", "600") or "600")
JOB_TIMEOUT_SECONDS = int(os.getenv("FOUNDRY_JOB_TIMEOUT_SECONDS", "14400") or "14400")
POLL_INTERVAL_SECONDS = int(os.getenv("FOUNDRY_POLL_INTERVAL_SECONDS", "60") or "60")
EVALUATION_LIMIT = int(os.getenv("FOUNDRY_EVALUATION_LIMIT", "20") or "20")
print({
    "paid_sft": RUN_PAID_SFT,
    "paid_rft": RUN_PAID_RFT,
    "live_evaluation": RUN_LIVE_EVALUATION,
    "use_live_generated_output": USE_LIVE_GENERATED_OUTPUT,
})


## 1. Immutable provenance and split membership

Every source dataset, tool definition, policy, OpenAPI document, and grader used below is byte-preserved. The manifest maps meaningful public filenames back to source paths and hashes. No record was rewritten or moved between train, validation, and evaluation splits.

In [ ]:
manifest = load_json(DATA / "manifest.json")
for entry in manifest["files"]:
    path = ROOT / entry["path"]
    assert path.stat().st_size == entry["bytes"], entry["path"]
    assert sha256_file(path) == entry["sha256"], entry["path"]
print(f"Validated {len(manifest['files'])} immutable inputs.")

## 2. Policy, OpenAPI, and tool surfaces

The natural-language policy requires identity verification, one customer per conversation, explicit confirmation before mutations, one tool call at a time, and transfer only when the request is out of scope. The OpenAPI document grounds synthetic generation. The 15-tool SFT/evaluation surface omits two RFT-only capabilities; the 17-tool rollout surface adds order listing and policy verification.

In [ ]:
policy = (DATA / "retail-policy.md").read_text(encoding="utf-8")
openapi = load_json(CONFIGS / "retail-openapi-with-policy.json")
sft_tools = load_json(CONFIGS / "sft-evaluation-tools.json")
rft_tools_template = load_json(CONFIGS / "rft-tools.json")

assert openapi["openapi"].startswith("3.")
assert len(sft_tools) == 15
assert len(rft_tools_template) == 17
print(policy.splitlines()[0])
print("OpenAPI paths:", len(openapi["paths"]))
print("SFT/eval tools:", [tool["function"]["name"] for tool in sft_tools])
print("RFT tools:", [tool["name"] for tool in rft_tools_template])

### MCP versus remote RFT tools

- **MCP agent runtime:** one MCP server URL advertises tools and may require per-call approval. Use HTTPS, managed identity/OAuth or a secret-store-backed token, explicit approval for mutations, bounded timeouts, and audit logs.
- **RFT rollout runtime:** each tool binding names a remote HTTPS endpoint. Resolve the placeholder only in memory from `FOUNDRY_TOOLS_SERVER_URL`; never commit resource names, regions, endpoints, or headers.

Authentication failures are non-retryable until credentials change. Retry 429/5xx/timeouts with bounded exponential backoff. Invalid arguments, policy denial, 4xx schema errors, and malformed outputs should terminate the affected step with an explicit failure signal. Never fabricate tool output.

In [ ]:
# Resolve real rollout endpoints only for a new paid RFT submission.
if RUN_PAID_RFT:
    tools_server_url = require_env("FOUNDRY_TOOLS_SERVER_URL")["FOUNDRY_TOOLS_SERVER_URL"]
    resolved_rft_tools = resolve_tool_endpoints(CONFIGS / "rft-tools.json", tools_server_url)
    tools_token = os.getenv("FOUNDRY_TOOLS_AUTH_TOKEN", "").strip()
    if tools_token:
        for tool in resolved_rft_tools:
            tool["headers"] = {"Authorization": f"Bearer {tools_token}"}
else:
    resolved_rft_tools = rft_tools_template
assert all("name" in tool and "server_url" in tool for tool in resolved_rft_tools)


## 3. Data contracts and exact live-output binding

The preserved synthetic snapshots are generation outputs, not recomputed approximations. The curated SFT splits remain the default because their membership differs from those snapshots. To train on a live generator result, set `FOUNDRY_USE_LIVE_GENERATED_OUTPUT=true` plus both `FOUNDRY_LIVE_GENERATED_TRAIN_PATH` and `FOUNDRY_LIVE_GENERATED_VALIDATION_PATH`. The notebook validates those exact files and passes those same resolved paths to upload; missing or invalid live output raises and never falls back.


In [ ]:
preserved_generated = validate_exact_generated_output(
    DATA / "synthetic-source-train.jsonl",
    DATA / "synthetic-source-validation.jsonl",
)
print(preserved_generated)

if USE_LIVE_GENERATED_OUTPUT:
    live_paths = require_env(
        "FOUNDRY_LIVE_GENERATED_TRAIN_PATH",
        "FOUNDRY_LIVE_GENERATED_VALIDATION_PATH",
    )
    selected_sft_splits = validate_exact_generated_output(
        Path(live_paths["FOUNDRY_LIVE_GENERATED_TRAIN_PATH"]),
        Path(live_paths["FOUNDRY_LIVE_GENERATED_VALIDATION_PATH"]),
    )
    sft_training_input = selected_sft_splits.train_path
    sft_validation_input = selected_sft_splits.validation_path
    sft_lineage = "live-generated-exact-output"
else:
    sft_training_input = (DATA / "sft-train.jsonl").resolve()
    sft_validation_input = (DATA / "sft-validation.jsonl").resolve()
    sft_lineage = "preserved-curated-splits"

print({
    "sft_lineage": sft_lineage,
    "training_path": str(sft_training_input),
    "validation_path": str(sft_validation_input),
})


In [ ]:
sft_train = load_jsonl(sft_training_input)
sft_validation = load_jsonl(sft_validation_input)
validate_conversation_rows(sft_train, label="SFT train")
validate_conversation_rows(sft_validation, label="SFT validation")

rft_train = load_jsonl(DATA / "rft-train.jsonl")
rft_validation = load_jsonl(DATA / "rft-validation.jsonl")
rft_required = (
    "messages", "tools", "reference_tool_calls",
    "reference_policy_args", "reference_policy_outcome",
)
validate_conversation_rows(rft_train, label="RFT train", required_fields=rft_required)
validate_conversation_rows(rft_validation, label="RFT validation", required_fields=rft_required)

print({
    "sft_train": len(sft_train),
    "sft_validation": len(sft_validation),
    "rft_train": len(rft_train),
    "rft_validation": len(rft_validation),
})

## 4. Deterministic next-tool evaluation

SFT should be measured on each next-tool decision using the preserved expanded evaluation set. The Python grader is order-invariant, gives exact tool-name/arguments full credit, gives name-only partial credit, and computes a soft F1 so extra or missing calls are penalized. It is transparent and does not incur model-grader variability.

In [ ]:
grader_path = ROOT / "graders" / "next-tool-call-grader.py"
spec = importlib.util.spec_from_file_location("next_tool_grader", grader_path)
next_tool_grader = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(next_tool_grader)

eval_rows = load_jsonl(DATA / "next-tool-evaluation.jsonl")
assert all("item" in row and "expected_output" in row["item"] for row in eval_rows)
example_expected = eval_rows[0]["item"]["expected_output"]["tool_calls"]
assert next_tool_grader.grade_tool_calls(example_expected, example_expected) == 1.0
assert next_tool_grader.grade_tool_calls([], example_expected) == 0.0
print("Evaluation decisions:", len(eval_rows))

## 5. SFT and RFT are complementary

| Technique | Learns from | Best fit here | Main risk |
|---|---|---|---|
| SFT | demonstrated assistant/tool traces | stable identity lookup, argument propagation, short policy sequences | imitation of dataset mistakes or narrow coverage |
| RFT | scalar reward over sampled rollouts | long multi-item return-policy reasoning and outcome quality | reward hacking, grader bias, tool instability, excessive episode length |

Use the same held-out business scenarios for final comparison, but do not tune against the test set.

In [ ]:
sft_job_plan = {
    "model": "${FOUNDRY_SFT_BASE_MODEL}",
    "training_file": str(sft_training_input),
    "validation_file": str(sft_validation_input),
    "method": {"type": "supervised"},
}

policy_grader = load_json(CONFIGS / "rft-policy-grader.json")
rft_job_plan = {
    "model": "${FOUNDRY_RFT_BASE_MODEL}",
    "training_file": str(DATA / "rft-train.jsonl"),
    "validation_file": str(DATA / "rft-validation.jsonl"),
    "method": {
        "type": "reinforcement",
        "reinforcement": {
            "grader": policy_grader,
            "tools": resolved_rft_tools,
            "max_episode_steps": 10,
            "hyperparameters": {
                "eval_interval": 1,
                "eval_samples": 2,
                "compute_multiplier": 1,
                "reasoning_effort": "medium",
                "n_epochs": 1,
                "batch_size": 10,
                "learning_rate_multiplier": 1,
            },
        },
    },
}
print(json.dumps({"sft": sft_job_plan, "rft": rft_job_plan}, indent=2)[:4000])


## 6. Guarded SFT and RFT upload, submission/reuse, and terminal monitoring

Each path is independently opt-in. Existing `FOUNDRY_SFT_JOB_ID` or `FOUNDRY_RFT_JOB_ID` values are monitored without creating a new job. New jobs require the corresponding `FOUNDRY_RUN_PAID_*` flag and reviewed model settings. Uploads are content-addressed and reused by exact filename; all file/job waits are bounded, and failed or cancelled terminal states raise explicitly.


In [ ]:
def run_fine_tuning_path(
    *,
    context,
    label,
    existing_job_id,
    allow_create,
    base_model_env,
    train_path,
    validation_path,
    method,
):
    if existing_job_id:
        job_id = existing_job_id
    elif not allow_create:
        return None
    else:
        base_model = require_env(base_model_env)[base_model_env]
        train_upload = upload_or_reuse_file(
            context.openai_client,
            demo_slug="retail-agent-capstone",
            purpose_name=f"{label}-training",
            path=train_path,
            timeout_seconds=FILE_TIMEOUT_SECONDS,
        )
        validation_upload = upload_or_reuse_file(
            context.openai_client,
            demo_slug="retail-agent-capstone",
            purpose_name=f"{label}-validation",
            path=validation_path,
            timeout_seconds=FILE_TIMEOUT_SECONDS,
        )
        submitted = create_or_reuse_fine_tuning_job(
            context.openai_client,
            model=base_model,
            training_file_id=train_upload.resource_id,
            validation_file_id=validation_upload.resource_id,
            suffix=f"retail-agent-{label}",
            method=method,
        )
        job_id = submitted.resource_id
    return monitor_fine_tuning_job(
        context.openai_client,
        job_id,
        poll_interval_seconds=POLL_INTERVAL_SECONDS,
        timeout_seconds=JOB_TIMEOUT_SECONDS,
    )

sft_job_id = os.getenv("FOUNDRY_SFT_JOB_ID", "").strip()
rft_job_id = os.getenv("FOUNDRY_RFT_JOB_ID", "").strip()
terminal_jobs = {}

if RUN_PAID_SFT or RUN_PAID_RFT or sft_job_id or rft_job_id:
    config = load_foundry_config(ROOT / ".env" if (ROOT / ".env").is_file() else None)
    with create_project_context(config) as context:
        sft_job = run_fine_tuning_path(
            context=context,
            label="sft",
            existing_job_id=sft_job_id,
            allow_create=RUN_PAID_SFT,
            base_model_env="FOUNDRY_SFT_BASE_MODEL",
            train_path=sft_training_input,
            validation_path=sft_validation_input,
            method={"type": "supervised"},
        )
        if sft_job is not None:
            terminal_jobs["sft"] = sft_job.status

        live_policy_grader = json.loads(json.dumps(policy_grader))
        configured_grader_model = os.getenv("FOUNDRY_RFT_GRADER_MODEL", "").strip()
        if configured_grader_model:
            live_policy_grader["model"] = configured_grader_model
        live_rft_method = {
            "type": "reinforcement",
            "reinforcement": {
                "grader": live_policy_grader,
                "tools": resolved_rft_tools,
                "max_episode_steps": 10,
                "hyperparameters": {
                    "eval_interval": 1,
                    "eval_samples": 2,
                    "compute_multiplier": 1,
                    "reasoning_effort": "medium",
                    "n_epochs": 1,
                    "batch_size": 10,
                    "learning_rate_multiplier": 1,
                },
            },
        }
        rft_job = run_fine_tuning_path(
            context=context,
            label="rft",
            existing_job_id=rft_job_id,
            allow_create=RUN_PAID_RFT,
            base_model_env="FOUNDRY_RFT_BASE_MODEL",
            train_path=DATA / "rft-train.jsonl",
            validation_path=DATA / "rft-validation.jsonl",
            method=live_rft_method,
        )
        if rft_job is not None:
            terminal_jobs["rft"] = rft_job.status
    print(terminal_jobs)
else:
    print("No job operation. Configure existing job IDs or explicitly enable a paid SFT/RFT path.")


## 6. Policy grader, rollout termination, and failure behavior

The model-based policy grader scores each referenced item only after exact order/item identity is established. It then checks eligibility and concrete policy justification, producing 0, 0.5, or 1 per item before averaging. This is deliberately stricter than tone or generic helpfulness.

Calibrate it with positive, negative, ambiguous, multi-item, and adversarial outputs. Track agreement with human labels and inspect drift when the grader model changes. Grader exceptions or invalid numeric output must receive the configured invalid grade, not pass.

Terminate a rollout on a policy-complete final response, a non-retryable tool/policy error, user-confirmation boundary, transfer, or `max_episode_steps`. A mutation request without explicit confirmation must stop before execution. Repeated identical calls, empty observations, and tool loops should be marked as failures.

In [ ]:
assert policy_grader["type"] == "score_model"
assert policy_grader["pass_threshold"] == 0.5
assert "reference_policy_outcome" in json.dumps(policy_grader)
print(policy_grader["name"], policy_grader["model"])

## 7. Representative preserved evidence

The figures below are copied from existing analysis outputs and contain no resource, endpoint, job, evaluation, or customer identifiers. They are historical teaching evidence, not a guarantee for a rerun.

In [ ]:
from IPython.display import Image, display

for asset in [
    "synthetic-tool-frequency.png",
    "synthetic-tool-sequence-heatmap.png",
    "base-model-pass-rate-comparison.png",
    "rft-score-progression.png",
    "rft-pass-rate-progression.png",
]:
    display(Image(filename=str(ASSETS / asset)))

In [ ]:
metrics = load_json(RESULTS / "representative-metrics.json")
metrics

The historical RFT curve is non-monotonic early, then improves from mean policy score `0.42966` at step 0 to `0.72148` at step 18; pass rate rises from 28% to 60% on 50 samples per checkpoint. Select checkpoints using held-out score, pass rate, scenario coverage, and failure inspection—not training reward alone.

## 9. Resolve deployments, invoke candidates, and run held-out next-tool evaluation

The current SDK discovers but does not create deployments. Create any base/SFT/RFT deployments through your approved management path, then set `FOUNDRY_MODEL_NAME`, `FOUNDRY_SFT_MODEL_NAME`, and/or `FOUNDRY_RFT_MODEL_NAME`. The held-out evaluator sends the preserved conversation and tool schemas, requires one next-tool prediction, compares normalized function name and arguments, and **does not execute the predicted tool**. This keeps mutation scenarios safe while measuring the model-policy decision boundary.


In [ ]:
held_out_rows = load_jsonl(DATA / "next-tool-evaluation.jsonl")

def predict_next_tool(context, deployment_name, row):
    item = row["item"]
    try:
        response = invoke_chat(
            context.openai_client,
            deployment_name=deployment_name,
            messages=item["messages"],
            tools=item["tools"],
            tool_choice="required",
            max_tokens=512,
        )
    except Exception as error:
        if "max_completion_tokens" not in str(error):
            raise
        response = context.openai_client.chat.completions.create(
            model=deployment_name,
            messages=item["messages"],
            tools=item["tools"],
            tool_choice="required",
            max_completion_tokens=4096,
            reasoning_effort="low",
        )
    return normalize_tool_call(response.choices[0].message)

held_out_evaluations = {}
if RUN_LIVE_EVALUATION:
    candidates = {
        "base": os.getenv("FOUNDRY_MODEL_NAME", "").strip(),
        "sft": os.getenv("FOUNDRY_SFT_MODEL_NAME", "").strip(),
        "rft": os.getenv("FOUNDRY_RFT_MODEL_NAME", "").strip(),
    }
    candidates = {label: name for label, name in candidates.items() if name}
    if not candidates:
        raise RuntimeError(
            "Set at least one of FOUNDRY_MODEL_NAME, FOUNDRY_SFT_MODEL_NAME, or FOUNDRY_RFT_MODEL_NAME"
        )
    config = load_foundry_config(ROOT / ".env" if (ROOT / ".env").is_file() else None)
    with create_project_context(config) as context:
        for label, deployment_name in candidates.items():
            ensure_deployment(
                context.project_client,
                deployment_name=deployment_name,
                timeout_seconds=600,
            )
            result = evaluate_next_tool_calls(
                held_out_rows[:EVALUATION_LIMIT],
                lambda row, name=deployment_name: predict_next_tool(context, name, row),
            )
            held_out_evaluations[label] = {
                "exact": result["exact"],
                "tool_name_match": result["tool_name_match"],
                "total": result["total"],
            }
    print(held_out_evaluations)
else:
    print("Held-out endpoint evaluation skipped; enable FOUNDRY_RUN_LIVE_EVALUATION after configuring caller-created deployments.")


## 10. Capstone integration plan

A final Foundry prompt agent should use the selected model, the preserved policy as instructions, and an MCP tool with approval required for mutations. Compare base/SFT/RFT candidates with identical prompts, tool server version, policy version, and evaluation data. Record model, dataset hashes, grader hash/model, tool schema version, endpoint version, sampling, region, and SDK/API versions in an experiment manifest.

Keep these boundaries explicit:

- **Read operations:** may be auto-approved after identity is established.
- **Mutation operations:** require a human confirmation turn before execution.
- **Authentication:** credentials come from identity/secret infrastructure, never prompt text.
- **Observability:** log correlation IDs, tool names, status/latency, termination reason, and grader components without logging secrets or unnecessary customer content.
- **Cleanup:** delete temporary agent versions only if this run created them; never overwrite or delete a pre-existing endpoint configuration.


In [ ]:
agent_integration_plan = {
    "instructions": "data/retail-policy.md",
    "model_candidates": ["base", "sft", "rft"],
    "mcp": {
        "server_url": "${FOUNDRY_MCP_SERVER_URL}",
        "require_approval_for_mutations": True,
        "authentication": "managed identity, OAuth, or secret-store-backed token",
    },
    "evaluation_contracts": [
        "next-tool deterministic grader",
        "item-level policy outcome grader",
        "tool error and latency diagnostics",
        "confirmation and transfer policy checks",
    ],
}
agent_integration_plan

## Conclusions and next curriculum step

This capstone preserves the original data, tool, policy, and grader bytes while providing complete guarded SFT and RFT workflows. Live generated SFT data is selected only through documented `FOUNDRY_*` settings and the exact validated files flow directly into content-addressed upload. Both training paths support existing-job reuse, bounded terminal monitoring, explicit deployment resolution, endpoint invocation, and held-out next-tool evaluation that never executes mutation calls.

**Next curriculum step:** continue to `05-data-and-distillation`, beginning with teacher-student basics, then code distillation, synthetic tool-use SFT, and agent traces to SFT.

*Variability footnote:* outputs, pass rates, rewards, latency, and selected checkpoints vary with model snapshot, sampling, dataset version, grader model, service/API version, region, quota, and remote tool behavior. Treat the preserved figures as representative evidence and rerun held-out evaluation for your environment.

> **Variability note:** Actual results may vary by model version, data, configuration, region availability, and service conditions.
